In [1]:
# ==========================================
# AI-POWERED FX RISK INTELLIGENCE PLATFORM
# WEEK 4 - SESSION 2
# OUT-OF-SAMPLE TESTING AND WALK-FORWARD
# VALIDATION
# ==========================================

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from arch import arch_model
import warnings
warnings.filterwarnings("ignore")

# Load existing feature dataset
df = pd.read_csv("../data/processed/ml_features.csv")

# Convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Sort chronologically
df = df.sort_values("Date").reset_index(drop=True)

# Inspect dataset
print("Dateset date:", df.shape)
print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())
print("Available columns:")
print(df.columns.tolist())
print("Dataset preview:")
display(df.head()) 


Dateset date: (1235, 25)
First date: 2022-01-26 00:00:00
Last date: 2026-10-27 00:00:00
Available columns:
['Date', 'GBPUSD_Return_1D', 'GBPUSD_Return_5D', 'GBPUSD_Volatility_5D', 'GBPUSD_Volatility_20D', 'GBPUSD_Volatility_60D', 'GBPUSD_Momentum_5D', 'GBPUSD_Momentum_20D', 'EURUSD_Return_1D', 'EURUSD_Return_5D', 'EURUSD_Volatility_5D', 'EURUSD_Volatility_20D', 'EURUSD_Volatility_60D', 'EURUSD_Momentum_5D', 'EURUSD_Momentum_20D', 'USDJPY_Return_1D', 'USDJPY_Return_5D', 'USDJPY_Volatility_5D', 'USDJPY_Volatility_20D', 'USDJPY_Volatility_60D', 'USDJPY_Momentum_5D', 'USDJPY_Momentum_20D', 'GBPUSD_Target_Volatility_5D', 'EURUSD_Target_Volatility_5D', 'USDJPY_Target_Volatility_5D']
Dataset preview:


,Date,GBPUSD_Return_1D,GBPUSD_Return_5D,GBPUSD_Volatility_5D,GBPUSD_Volatility_20D,GBPUSD_Volatility_60D,GBPUSD_Momentum_5D,GBPUSD_Momentum_20D,EURUSD_Return_1D,EURUSD_Return_5D,...,USDJPY_Return_1D,USDJPY_Return_5D,USDJPY_Volatility_5D,USDJPY_Volatility_20D,USDJPY_Volatility_60D,USDJPY_Momentum_5D,USDJPY_Momentum_20D,GBPUSD_Target_Volatility_5D,EURUSD_Target_Volatility_5D,USDJPY_Target_Volatility_5D
0,2022-01-26,0.001594,-0.006580,0.002668,0.003079,0.003862,-0.006580,0.005769,-0.001730,-0.002216,...,-0.001193,-0.005927,0.001904,0.002915,0.003395,-0.005927,-0.007824,0.004777,0.006427,0.004670
1,2022-01-27,-0.003165,-0.010330,0.002516,0.002972,0.003805,-0.010330,-0.002280,-0.005397,-0.008804,...,0.007429,0.003271,0.004222,0.003381,0.003527,0.003271,-0.001975,0.004463,0.005866,0.003583
2,2022-01-28,-0.005918,-0.015396,0.002883,0.003243,0.003462,-0.015396,-0.008153,-0.008583,-0.014736,...,0.005500,0.011922,0.004057,0.003600,0.003583,0.011922,0.002512,0.001765,0.004519,0.003382
3,2022-01-31,0.001126,-0.011246,0.003447,0.003223,0.003463,-0.011246,-0.008793,0.000480,-0.016673,...,-0.000780,0.012981,0.003805,0.003603,0.003558,0.012981,0.001042,0.003996,0.004230,0.003635
4,2022-02-01,0.003133,-0.003254,0.003761,0.003263,0.003413,-0.003254,-0.002810,0.006929,-0.008344,...,-0.001223,0.009736,0.004184,0.003595,0.003551,0.009736,-0.001795,0.003984,0.004947,0.003650


In [2]:
# Identify volatility target columns
target_columns = [col for col in df.columns if "Target_Volatility" in col]
print("Available target variables:")
for col in target_columns:
    print(col)

Available target variables:
GBPUSD_Target_Volatility_5D
EURUSD_Target_Volatility_5D
USDJPY_Target_Volatility_5D


In [3]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate dates
print("Duplicate dates:", df["Date"].duplicated().sum())

# Check dataset dimensions
print("Observations:", len(df))
print("Features:", len(df.columns))

Missing values:
Date                           0
GBPUSD_Return_1D               0
GBPUSD_Return_5D               0
GBPUSD_Volatility_5D           0
GBPUSD_Volatility_20D          0
GBPUSD_Volatility_60D          0
GBPUSD_Momentum_5D             0
GBPUSD_Momentum_20D            0
EURUSD_Return_1D               0
EURUSD_Return_5D               0
EURUSD_Volatility_5D           0
EURUSD_Volatility_20D          0
EURUSD_Volatility_60D          0
EURUSD_Momentum_5D             0
EURUSD_Momentum_20D            0
USDJPY_Return_1D               0
USDJPY_Return_5D               0
USDJPY_Volatility_5D           0
USDJPY_Volatility_20D          0
USDJPY_Volatility_60D          0
USDJPY_Momentum_5D             0
USDJPY_Momentum_20D            0
GBPUSD_Target_Volatility_5D    0
EURUSD_Target_Volatility_5D    0
USDJPY_Target_Volatility_5D    0
dtype: int64
Duplicate dates: 0
Observations: 1235
Features: 25


In [4]:
# Define target varaibles
target_columns = ["GBPUSD_Target_Volatility_5D","EURUSD_Target_Volatility_5D","USDJPY_Target_Volatility_5D"]

# Identify predictive features
feature_columns = [col for col in df.columns if col not in ["Date"] + target_columns]

# Keep numerical predictive features only
feature_columns = df[feature_columns].select_dtypes(include=[np.number]).columns.tolist()

# Remove observations with missing features or targets
df_model = df.dropna(subset=feature_columns + target_columns).copy()

# Define independent variables
X = df_model[feature_columns]

# Define dependent varaibles for all three currency pairs
y = {"GBPUSD": df_model["GBPUSD_Target_Volatility_5D"], "EURUSD": df_model["EURUSD_Target_Volatility_5D"], "USDJPY": df_model["USDJPY_Target_Volatility_5D"]}

# Extract corresponding dates
dates = df_model["Date"]

In [5]:
# Verify modelling datatset
print("Feature shape:", X.shape)
print("Target shapes:")
for currency, target in y.items():
    print(f"{currency}: {target.shape}")
print("First date:", dates.min())
print("Last date:", dates.max())
print("Number of features:", X.shape[1])
print("Number of observations:", len(X))
print("Missing values:")
print("Features:", X.isnull().sum().sum())
for currency, target in y.items():
    print(f"{currency}:", target.isnull().sum())
print("Feature columns:")
print(feature_columns)

Feature shape: (1235, 21)
Target shapes:
GBPUSD: (1235,)
EURUSD: (1235,)
USDJPY: (1235,)
First date: 2022-01-26 00:00:00
Last date: 2026-10-27 00:00:00
Number of features: 21
Number of observations: 1235
Missing values:
Features: 0
GBPUSD: 0
EURUSD: 0
USDJPY: 0
Feature columns:
['GBPUSD_Return_1D', 'GBPUSD_Return_5D', 'GBPUSD_Volatility_5D', 'GBPUSD_Volatility_20D', 'GBPUSD_Volatility_60D', 'GBPUSD_Momentum_5D', 'GBPUSD_Momentum_20D', 'EURUSD_Return_1D', 'EURUSD_Return_5D', 'EURUSD_Volatility_5D', 'EURUSD_Volatility_20D', 'EURUSD_Volatility_60D', 'EURUSD_Momentum_5D', 'EURUSD_Momentum_20D', 'USDJPY_Return_1D', 'USDJPY_Return_5D', 'USDJPY_Volatility_5D', 'USDJPY_Volatility_20D', 'USDJPY_Volatility_60D', 'USDJPY_Momentum_5D', 'USDJPY_Momentum_20D']


In [6]:
# Chronological train/test split

# Number of observations
n = len(X)

# Define validation parameters
initial_train_size = int(n * 0.70)
test_size = int(n * 0.10)
gap = 5

# Display initial dataset configuration
print("Total observations:", n)

print("Initial training period:")
print("Observations:", initial_train_size)
print("Start date:", dates.iloc[0])
print("End date:", dates.iloc[initial_train_size - 1])

print("Testing window:")
print("Observations per window:", test_size)

print("Validation configuration:")
print("Inital training proportion: 70%")
print("Testing window proportion: 10%")
print("Number of testing periods: 3")
print("Gap between training and testing:", gap)

Total observations: 1235
Initial training period:
Observations: 864
Start date: 2022-01-26 00:00:00
End date: 2025-05-22 00:00:00
Testing window:
Observations per window: 123
Validation configuration:
Inital training proportion: 70%
Testing window proportion: 10%
Number of testing periods: 3
Gap between training and testing: 5


In [7]:
# Expanding window valiation

# Store validation periods
validation_periods = []
for fold in range(3):

    # Calculate testing period boundaires
    test_start = initial_train_size + (fold * test_size)
    test_end = test_start + test_size

    # Apply five-observation gap
    train_end = test_start - gap

    # Create training and testing indices
    train_indices = np.arange(0, train_end)
    test_indices = np.arange(test_start, test_end)

    # Store period information
    validation_periods.append({"fold": fold + 1, "train_indices": train_indices, "test_indices": test_indices})

# Display validation periods
for period in validation_periods:
    train_idx = period["train_indices"]
    test_idx = period["test_indices"]

    print(f"Validation period {period['fold']}")
    print("-" * 40)

    print("Training observations:", len(train_idx))
    print("Training start:", dates.iloc[train_idx[0]])
    print("Training end:", dates.iloc[train_idx[-1]])

    print("Testing observations:", len(test_idx))
    print("Testing start:", dates.iloc[test_idx[0]])
    print("Training end:", dates.iloc[test_idx[-1]])

Validation period 1
----------------------------------------
Training observations: 859
Training start: 2022-01-26 00:00:00
Training end: 2025-05-15 00:00:00
Testing observations: 123
Testing start: 2025-05-23 00:00:00
Training end: 2025-11-11 00:00:00
Validation period 2
----------------------------------------
Training observations: 982
Training start: 2022-01-26 00:00:00
Training end: 2025-11-04 00:00:00
Testing observations: 123
Testing start: 2025-11-12 00:00:00
Training end: 2026-05-05 00:00:00
Validation period 3
----------------------------------------
Training observations: 1105
Training start: 2022-01-26 00:00:00
Training end: 2026-04-28 00:00:00
Testing observations: 123
Testing start: 2026-05-06 00:00:00
Training end: 2026-10-23 00:00:00


In [8]:
# Configure forecasting models

# Define Random Forest
rf_model = RandomForestRegressor(n_estimators=200,max_depth=6,min_samples_split=10,min_samples_leaf=5,random_state=42,n_jobs=1)

# Define XGBoost 
xgb_model = XGBRegressor(n_estimators=200,max_depth=3,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,object="reg:squarederror",random_state=42,n_jobs=1)

# Store model configurations
models = {"Random Forest": rf_model, "XGBoost": xgb_model}
print("Forecasting models configured successfully.")
for model_name in models:
    print(model_name)

Forecasting models configured successfully.
Random Forest
XGBoost


In [9]:
# Walk-Forward validation
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
walk_forward_results = []
performance_results = []

for currency, target in y.items():
    print(f"\nProcessing {currency}")
    print("=" * 60)
    for model_name, model in models.items():
        for period in validation_periods:
            fold = period["fold"]

            train_idx = period["train_indices"]
            test_idx = period["test_indices"]

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = target.iloc[train_idx]
            y_test = target.iloc[test_idx]

            # Train fresh model
            fitted_model = clone(model)
            fitted_model.fit(X_train, y_train)

            # Generate predictions
            predictions = fitted_model.predict(X_test)

            # Calculate metrics
            mae = mean_absolute_error(y_test, predictions)

            rmse = np.sqrt(mean_squared_error(y_test, predictions))

            r2 = fitted_model.score(X_test, y_test)

            # Store performance metrics
            performance_results.append({
                "Currency": currency,
                "Model": model_name,
                "Fold": fold,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "Unique_Predictions": np.unique(predictions).size
            })

            # Store individual predictions
            for i, idx in enumerate(test_idx):

                walk_forward_results.append({
                    "Currency": currency,
                    "Model": model_name,
                    "Fold": fold,
                    "Date": dates.iloc[idx],
                    "Actual": y_test.iloc[i],
                    "Predicted": predictions[i]
                })

            print(
                f"{currency} | {model_name} | "
                f"Fold {fold} | "
                f"MAE: {mae:.6f} | "
                f"RMSE: {rmse:.6f} | "
                f"R2: {r2:.4f} | "
                f"Unique predictions: "
                f"{np.unique(predictions).size}"
            )

# Create result DataFrames
walk_forward_df = pd.DataFrame(walk_forward_results)

performance_df = pd.DataFrame(performance_results)

# Verify results
print("\nValidation completed")

print("Total predictions:", len(walk_forward_df))

print("\nPerformance summary:")
display(performance_df)

print("\nPrediction counts:")
print(
    walk_forward_df.groupby(
        ["Currency", "Model", "Fold"]
    ).size()
)



Processing GBPUSD
GBPUSD | Random Forest | Fold 1 | MAE: 0.001360 | RMSE: 0.001658 | R2: -0.4986 | Unique predictions: 123
GBPUSD | Random Forest | Fold 2 | MAE: 0.001266 | RMSE: 0.001612 | R2: -0.0925 | Unique predictions: 123
GBPUSD | Random Forest | Fold 3 | MAE: 0.001449 | RMSE: 0.001661 | R2: -0.6112 | Unique predictions: 123
GBPUSD | XGBoost | Fold 1 | MAE: 0.001466 | RMSE: 0.001807 | R2: -0.7815 | Unique predictions: 123
GBPUSD | XGBoost | Fold 2 | MAE: 0.001237 | RMSE: 0.001593 | R2: -0.0670 | Unique predictions: 123
GBPUSD | XGBoost | Fold 3 | MAE: 0.001357 | RMSE: 0.001591 | R2: -0.4789 | Unique predictions: 123

Processing EURUSD
EURUSD | Random Forest | Fold 1 | MAE: 0.001649 | RMSE: 0.002108 | R2: -0.1511 | Unique predictions: 123
EURUSD | Random Forest | Fold 2 | MAE: 0.001687 | RMSE: 0.001904 | R2: -0.2021 | Unique predictions: 123
EURUSD | Random Forest | Fold 3 | MAE: 0.001492 | RMSE: 0.001715 | R2: -2.8033 | Unique predictions: 123
EURUSD | XGBoost | Fold 1 | MAE: 0.

,Currency,Model,Fold,MAE,RMSE,R2,Unique_Predictions
0,GBPUSD,Random Forest,1,0.001360,0.001658,-0.498617,123
1,GBPUSD,Random Forest,2,0.001266,0.001612,-0.092534,123
2,GBPUSD,Random Forest,3,0.001449,0.001661,-0.611165,123
3,GBPUSD,XGBoost,1,0.001466,0.001807,-0.781455,123
4,GBPUSD,XGBoost,2,0.001237,0.001593,-0.066960,123
5,GBPUSD,XGBoost,3,0.001357,0.001591,-0.478920,123
6,EURUSD,Random Forest,1,0.001649,0.002108,-0.151094,123
7,EURUSD,Random Forest,2,0.001687,0.001904,-0.202122,123
8,EURUSD,Random Forest,3,0.001492,0.001715,-2.803275,123
9,EURUSD,XGBoost,1,0.001698,0.002255,-0.317718,123



Prediction counts:
Currency  Model          Fold
EURUSD    Random Forest  1       123
                         2       123
                         3       123
          XGBoost        1       123
                         2       123
                         3       123
GBPUSD    Random Forest  1       123
                         2       123
                         3       123
          XGBoost        1       123
                         2       123
                         3       123
USDJPY    Random Forest  1       123
                         2       123
                         3       123
          XGBoost        1       123
                         2       123
                         3       123
dtype: int64


In [10]:
import os

# Create evaluation directory if it does not exist
os.makedirs("../outputs/evaluation", exist_ok=True)

# Save individual walk-forward predictions
walk_forward_df.to_csv("../outputs/evaluation/walk_forward_results.csv",index=False)

# Save model performance metrics
performance_df.to_csv("../outputs/evaluation/walk_forward_performance.csv",index=False)
print("Session 2 results successfully saved.")
print("\nWalk-forward predictions:")
print(walk_forward_df.shape)
print("\nPerformance results:")
print(performance_df.shape)

Session 2 results successfully saved.

Walk-forward predictions:
(2214, 6)

Performance results:
(18, 7)
